In [1]:
import tensorflow as tf
import time
import numpy as np
import os
import copy
import pickle5 as pickle
import argparse
import utilityarm as utility
import pandas as pd
from sklearn.metrics import *
import tensorflow.keras.backend as K

In [2]:
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior() 

class FairAdvBPR:

    def __init__(self, sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count):
       
        self.dataname = dict_args['dataname']

        self.key_type = key_type
        self.user_type_list = user_type_list
        self.item_type_count = item_type_count
        self.layers = dict_args['layers']
        self.sess = sess
        
        self.num_cols = len(train_df['item_id'].unique())
        self.num_rows = len(train_df['user_id'].unique())

        self.hidden_neuron = dict_args['hidden_neuron']
        self.neg = dict_args['neg']
        self.batch_size = dict_args['batch_size']

        self.train_df = train_df
        self.vali_df = test_df
        self.num_train = len(self.train_df)
        self.num_vali = len(self.vali_df)

        self.train_epoch = dict_args['train_epoch']
        self.train_epoch_a = dict_args['train_epoch_a']

        self.lr_r = dict_args['lr_r'] # learning rate
        self.lr_a = dict_args['lr_a'] # learning rate
        self.alpha = dict_args['alpha'] # learning rate
        self.optimizer_method = dict_args['optimizer_method']
        self.display_step = dict_args['display_step']
        
        self.type_error_weight = type_error_weight
        self.num_type = dict_args['num_type']
        
        self.user_type = user_type
        self.type_count_list = []
        for k in range(self.num_type):
            self.type_count_list.append(np.sum(user_type[:,k]))

        
        self.reg = dict_args['reg'] # regularization term trade-off
        self.reg_s = dict_args['reg_s']

        print('**********fairAdvBPR**********')
        #print(self.args)
        self._prepare_model()

    def loadmodel(self, saver, checkpoint_dir):
        ckpt = tf.train.get_checkpoint_state(checkpoint_dir)
        if ckpt and ckpt.model_checkpoint_path:
            ckpt_name = os.path.basename(ckpt.model_checkpoint_path)
            saver.restore(self.sess, os.path.join(checkpoint_dir, ckpt_name))
            return True
        else:
            return False
        
    def run(self):
        init = tf.global_variables_initializer()
        self.sess.run(init)

        saver = tf.train.Saver([self.P, self.Q])
        self.loadmodel(saver, "./"+self.dataname+"/BPR_check_points")

        for epoch_itr in range(1, self.train_epoch + 1 + self.train_epoch_a):
            self.train_model(epoch_itr)
            if epoch_itr % self.display_step == 0:
                self.test_model(epoch_itr)
        return self.make_records()

    def _prepare_model(self):
        with tf.name_scope("input_data"):
            self.user_input = tf.placeholder(tf.int32, shape=[None, 1], name="user_input")
            self.item_input_pos = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_pos")
            self.item_input_neg = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_neg")

            self.input_user_type = tf.placeholder(dtype=tf.float32, shape=[None, self.num_type]
                                                   , name="input_user_type")
            self.input_user_error_weight = tf.placeholder(dtype=tf.float32, shape=[None, 1]
                                                          , name="input_user_error_weight")

        with tf.variable_scope("BPR", reuse=tf.AUTO_REUSE):
            self.P = tf.get_variable(name="P",
                                     initializer=tf.truncated_normal(shape=[self.num_rows, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
            self.Q = tf.get_variable(name="Q",
                                     initializer=tf.truncated_normal(shape=[self.num_cols+1, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
        para_r = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="BPR")

        with tf.variable_scope("Adversarial", reuse=tf.AUTO_REUSE):
            num_layer = len(self.layers)
            adv_W = []
            adv_b = []
            for l in range(num_layer):
                if l == 0:
                    in_shape = 1
                else:
                    in_shape = self.layers[l - 1]
                adv_W.append(tf.get_variable(name="adv_W" + str(l),
                                             initializer=tf.truncated_normal(shape=[in_shape, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
                adv_b.append(tf.get_variable(name="adv_b" + str(l),
                                             initializer=tf.truncated_normal(shape=[1, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
            adv_W_out = tf.get_variable(name="adv_W_out",
                                        initializer=tf.truncated_normal(shape=[self.layers[-1], self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)

            adv_b_out = tf.get_variable(name="adv_b_out",
                                        initializer=tf.truncated_normal(shape=[1, self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)
        para_a = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="Adversarial")

        p = tf.reduce_sum(tf.nn.embedding_lookup(self.P, self.user_input), 1)
        q_neg = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_neg), 1)
        q_pos = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_pos), 1)

        predict_pos = tf.reduce_sum(p * q_pos, 1)
        predict_neg = tf.reduce_sum(p * q_neg, 1)

        r_cost1 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg)))
        r_cost2 = self.reg * 0.5 * (self.l2_norm(self.P) + self.l2_norm(self.Q))  # regularization term
        pred = tf.matmul(self.P, tf.transpose(self.Q))
        self.s_mean = tf.reduce_mean(pred, axis=1)
        self.s_std = tf.keras.backend.std(pred, axis=1)
        self.s_cost = tf.reduce_sum(tf.square(self.s_mean) + tf.square(self.s_std) - 2 * tf.log(self.s_std) - 1)
        self.r_cost = r_cost1 + r_cost2 + self.reg_s * 0.5 * self.s_cost

        adv_last = tf.reshape(predict_pos, [tf.shape(self.input_user_type)[0], 1])
        for l in range(num_layer):
            adv = tf.nn.relu(tf.matmul(adv_last, adv_W[l]) + adv_b[l])
            adv_last = adv
        self.adv_output = tf.nn.sigmoid(tf.matmul(adv_last, adv_W_out) + adv_b_out)
        self.a_cost = tf.reduce_sum(tf.square(self.adv_output - self.input_user_type) * self.input_user_error_weight)

        self.all_cost = self.r_cost - self.alpha * self.a_cost  # the loss function

        with tf.variable_scope("Optimizer", reuse=tf.AUTO_REUSE):
            self.r_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.r_cost, var_list=para_r)
            self.a_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_a).minimize(self.a_cost, var_list=para_a)
            self.all_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.all_cost, var_list=para_r)


    def train_model(self, itr):
        NS_start_time = time.time() * 1000.0
        epoch_r_cost = 0.0
        epoch_s_cost = 0.0
        epoch_s_mean = 0.0
        epoch_s_std = 0.0
        epoch_a_cost = 0.0
        num_sample, user_list, item_pos_list, item_neg_list = utility.negative_sample(self.train_df, self.num_rows,
                                                                                      self.num_cols, self.neg)
        NS_end_time = time.time() * 1000.0

        start_time = time.time() * 1000.0
        num_batch = int(num_sample / float(self.batch_size)) + 1
        random_idx = np.random.permutation(num_sample)
        for i in range(num_batch):
            # get the indices of the current batch
            if i == num_batch - 1:
                batch_idx = random_idx[i * self.batch_size:]
            elif i < num_batch - 1:
                batch_idx = random_idx[(i * self.batch_size):((i + 1) * self.batch_size)]

            if itr > self.train_epoch:
                random_idx_a = np.random.permutation(num_sample)
                print("boucle adversarial debut-- num batch ",i)
                for j in range(num_batch):
                    if j == num_batch - 1:
                        batch_idx_a = random_idx_a[j * self.batch_size:]
                    elif j < num_batch - 1:
                        batch_idx_a = random_idx_a[(j * self.batch_size):((j + 1) * self.batch_size)]
                    user_idx_list = ((user_list[batch_idx_a, :]).reshape((len(batch_idx_a)))).tolist()
                    _, tmp_a_cost = self.sess.run(  # do the optimization by the minibatch
                        [self.a_optimizer, self.a_cost],
                        feed_dict={self.user_input: user_list[batch_idx_a, :],
                                   self.item_input_pos: item_pos_list[batch_idx_a, :],
                                   self.item_input_neg: item_neg_list[batch_idx_a, :],
                                   self.input_user_type: self.user_type[user_idx_list,:],
                                   self.input_user_error_weight: self.type_error_weight[user_idx_list,:]})
                    epoch_a_cost += tmp_a_cost

                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost, tmp_s_cost, tmp_s_mean, tmp_s_std = self.sess.run(  # do the optimization by the minibatch
                    [self.all_optimizer, self.all_cost, self.s_cost, self.s_mean, self.s_std],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
                epoch_s_mean += np.mean(tmp_s_mean)
                epoch_s_std += np.mean(tmp_s_std)
                epoch_s_cost += tmp_s_cost
                print("boucle adversarial fin")
            else:
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost, tmp_s_cost, tmp_s_mean, tmp_s_std = self.sess.run(  # do the optimization by the minibatch
                    [self.r_optimizer, self.r_cost, self.s_cost, self.s_mean, self.s_std],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
                epoch_s_mean += np.mean(tmp_s_mean)
                epoch_s_std += np.mean(tmp_s_std)
                epoch_s_cost += tmp_s_cost
        epoch_a_cost /= num_batch
        if itr % self.display_step == 0:
            print ("Training //", "Epoch %d //" % itr, " Total r_cost = %.5f" % epoch_r_cost,
                   " Total s_cost = %.5f" % epoch_s_cost,
                   " Total s_mean = %.5f" % epoch_s_mean,
                   " Total s_std = %.5f" % epoch_s_std,
                   " Total a_cost = %.5f" % epoch_a_cost,
                   "Training time : %d ms" % (time.time() * 1000.0 - start_time),
                   "negative Sampling time : %d ms" % (NS_end_time - NS_start_time),
                   "negative samples : %d" % (num_sample))
       
    def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
        if itr % self.display_step == 0:
            start_time = time.time() * 1000.0
            P, Q = self.sess.run([self.P, self.Q])
            Rec = np.matmul(P, Q.T)

            [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
#             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_genre, self.item_genre_list,
#                                      self.user_genre_count)
            utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
            auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
            print("AUC global is: ", auc)

            filename = './fairAdvBPR_results/epoch'+ str(itr) +'_Rec_' + self.dataname + '_fairAdvBPR.npy'
            os.makedirs(os.path.dirname(filename), exist_ok=True)           
            with open(filename, "wb") as f:
                np.save(f, Rec)
            

    def make_records(self):  # record all the results' details into files
        P, Q = self.sess.run([self.P, self.Q])
        Rec = np.matmul(P, Q.T)

        [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
        return precision, recall, f_score, NDCG, Rec

#     def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
#         if itr % self.display_step == 0:
#             start_time = time.time() * 1000.0
#             P, Q = self.sess.run([self.P, self.Q])
#             Rec = np.matmul(P, Q.T)

#             [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
# #             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_type, self.user_type_list,
# #                                      self.item_type_count)
#             utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
#             auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
#             print("AUC global is: ", auc)
#             print (
#                 "Testing //", "Epoch %d //" % itr,
#                 "Testing time : %d ms" % (time.time() * 1000.0 - start_time))
#             print("=" * 200)


    @staticmethod
    def l2_norm(tensor):
        return tf.reduce_sum(tf.square(tensor))


Instructions for updating:
non-resource variables are not supported in the long term


In [3]:

#optimizer_method = ['Adam', 'Adadelta', 'Adagrad', 'RMSProp', 'GradientDescent','Momentum'], default='Adam')


train_epoch = 1
train_epoch_a = 10 #default 20
display_step = 1
lr_r = 0.01
lr_a = 0.005
reg = 0.1
reg_s = 30
alpha = 1000
optimizer_method = 'Adam'
hidden_neuron = 20
n = 1
neg = 5
batch_size = 1024
layers = [50, 50, 50, 50]
dataname = 'modcloth'

In [4]:
# parser.add_argument('--train_epoch', type=int, default=0)
# parser.add_argument('--train_epoch_a', type=int, default=20)
# parser.add_argument('--display_step', type=int, default=1)
# parser.add_argument('--lr_r', type=float, default=0.01)
# parser.add_argument('--lr_a', type=float, default=0.005)
# parser.add_argument('--reg', type=float, default=0.1)
# parser.add_argument('--reg_s', type=float, default=30)
# parser.add_argument('--hidden_neuron', type=int, default=20)
# parser.add_argument('--n', type=int, default=1)
# parser.add_argument('--neg', type=int, default=5)
# parser.add_argument('--alpha', type=float, default=1000.0)
# parser.add_argument('--batch_size', type=int, default=1024)
# parser.add_argument('--layers', nargs='?', default='[50, 50, 50, 50]')
# parser.add_argument('--dataname', nargs='?', default='ml1m-6')

In [5]:
dict_args =  {"train_epoch": train_epoch,
              "train_epoch_a": train_epoch_a,
            "display_step":display_step,
            "lr_r":lr_r,
            "lr_a":lr_a,
            "reg":reg,
            "reg_s":reg_s,
            "alpha":alpha,
            "optimizer_method":optimizer_method,
            "hidden_neuron":hidden_neuron,
            "n":n,
            "neg":neg,
            "batch_size":batch_size,
            "layers":layers,
            "dataname":dataname}
dict_args

{'train_epoch': 1,
 'train_epoch_a': 10,
 'display_step': 1,
 'lr_r': 0.01,
 'lr_a': 0.005,
 'reg': 0.1,
 'reg_s': 30,
 'alpha': 1000,
 'optimizer_method': 'Adam',
 'hidden_neuron': 20,
 'n': 1,
 'neg': 5,
 'batch_size': 1024,
 'layers': [50, 50, 50, 50],
 'dataname': 'modcloth'}

In [6]:
with open('./training_df_modcloth.pkl', 'rb') as f:
    train_df = pickle.load(f,encoding='latin1')

# with open('./' + dataname + '/valiing_df.pkl', 'rb') as f:
#     vali_df = pickle.load(f,encoding='latin1')  # for validation
    
with open('./testing_df_modcloth.pkl', 'rb') as f:
    test_df = pickle.load(f,encoding='latin1')  # for validation
# vali_df = pickle.load(open('./' + dataname + '/testing_df.pkl'))  # for testing

with open('./key_type_modcloth.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')
    
with open('./user_idd_type_list_modcloth.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')
    
with open('./type_user_vector_modcloth.pkl', 'rb') as f:
    type_user_vector = pickle.load(f,encoding='latin1')

with open('./type_count_modcloth.pkl', 'rb') as f:
    type_count = pickle.load(f,encoding='latin1')
    
with open('./item_type_count_modcloth.pkl', 'rb') as f:
    item_type_count = pickle.load(f,encoding='latin1')

In [7]:
train_df.head(20)

,user_id,item_id,rating
0,0,0,4
1,1,0,4
2,2,0,4
3,3,0,2
4,4,0,4
5,5,0,4
6,6,0,5
7,7,0,5
8,8,0,5
9,9,0,5


In [8]:
train_df.shape

(40354, 3)

In [9]:
test_df.head(20)

,user_id,item_id,rating
0,17,1,5
1,20,0,5
2,21,0,2
3,26,2,5
4,32,2,5
5,39,1,3
6,45,0,4
7,56,0,5
8,60,2,5
9,61,2,4


In [10]:
test_df.shape

(10894, 3)

In [11]:
print(len(user_idd_type_list))

6339


In [12]:
user_idd_type_list

[['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Large'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Large'],
 ['Large'],
 ['Large'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Large'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Small'],
 ['Large'],
 ['Small'],
 ['Small'],
 ['S

In [13]:
type_user_vector

{'Small': array([[1., 1., 1., ..., 0., 0., 1.]]),
 'Large': array([[0., 0., 0., ..., 1., 1., 0.]])}

In [14]:
len(item_type_count)

915

In [15]:
item_type_count

[{'Small': 4314, 'Large': 1684},
 {'Small': 4317, 'Large': 1678},
 {'Small': 4273, 'Large': 1660},
 {'Small': 4121, 'Large': 1628},
 {'Small': 4324, 'Large': 1673},
 {'Small': 4338, 'Large': 1676},
 {'Small': 4458, 'Large': 1697},
 {'Small': 4318, 'Large': 1677},
 {'Small': 4385, 'Large': 1704},
 {'Small': 4419, 'Large': 1682},
 {'Small': 4332, 'Large': 1671},
 {'Small': 4248, 'Large': 1662},
 {'Small': 4242, 'Large': 1669},
 {'Small': 4403, 'Large': 1680},
 {'Small': 4224, 'Large': 1643},
 {'Small': 4284, 'Large': 1654},
 {'Small': 4376, 'Large': 1692},
 {'Small': 4338, 'Large': 1644},
 {'Small': 4413, 'Large': 1697},
 {'Small': 4087, 'Large': 1576},
 {'Small': 4451, 'Large': 1670},
 {'Small': 4525, 'Large': 1693},
 {'Small': 4292, 'Large': 1645},
 {'Small': 4403, 'Large': 1687},
 {'Small': 4283, 'Large': 1649},
 {'Small': 4402, 'Large': 1699},
 {'Small': 4500, 'Large': 1698},
 {'Small': 4446, 'Large': 1683},
 {'Small': 4462, 'Large': 1687},
 {'Small': 4449, 'Large': 1700},
 {'Small':

In [16]:
print(type_count)

[('Small', 4614), ('Large', 1725)]


In [17]:
num_item = len(train_df['item_id'].unique())
num_user = len(train_df['user_id'].unique())
num_type = len(key_type)
print('items number : ',num_item)
print('users number : ',num_user)
print('user types : ',key_type)


items number :  915
users number :  6339
user types :  ['Small', 'Large']


In [18]:
dict_args["num_type"] = len(key_type)

In [19]:
user_type_list = [] #preprocessing to be sure that user types are really the right ones armielle 
for u in range(num_user):
    gl = user_idd_type_list[u]
    tmp = []
    for g in gl:
        if g in key_type:
            tmp.append(g)
    user_type_list.append(tmp)

print(len(user_type_list))

6339


In [20]:
# genreate user_type matrix
user_type = np.zeros((num_user, num_type))
for u in range(num_user):
    gl = user_type_list[u]
    for k in range(num_type):
        if key_type[k] in gl:
            user_type[u, k] = 1.0

In [21]:
len(user_type_list)

6339

In [22]:
print('*' * 50)
print('number of positive feedback: ' + str(len(train_df)))
print('estimated number of training samples: ' + str(neg * len(train_df)))
print('*' * 50)

**************************************************
number of positive feedback: 40354
estimated number of training samples: 201770
**************************************************


In [23]:
key_type

['Small', 'Large']

In [24]:
type_count

[('Small', 4614), ('Large', 1725)]

In [25]:
type_count = dict(type_count)

In [26]:
type_count_mean_reciprocal = []
for k in key_type:
    type_count_mean_reciprocal.append(1.0 / type_count[k])
type_count_mean_reciprocal = (np.array(type_count_mean_reciprocal)).reshape((num_type, 1))
type_error_weight = np.dot(user_type, type_count_mean_reciprocal)


In [27]:
# generate user_type matrix
type_user_indicator = np.zeros((num_type, num_user))

for k in range(num_type):
    type_user_indicator[k,:] = type_user_vector[key_type[k]]


In [28]:
precision = np.zeros(4)
recall = np.zeros(4)
f1 = np.zeros(4)
ndcg = np.zeros(4)
RSP = np.zeros(4)
REO = np.zeros(4)

precision 

array([0., 0., 0., 0.])

In [29]:
len(user_type_list)

6339

In [ ]:
#tf.compat.v1.disable_eager_execution()

for i in range(n):
    #with tf.compat.v1.Session() as sess:
    with tf.Session() as sess:
        fairadvbpr = FairAdvBPR(sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count)
        [prec_one, rec_one, f_one, ndcg_one, Rec] = fairadvbpr.run()
       

**********fairAdvBPR**********
INFO:tensorflow:Restoring parameters from ./modcloth/BPR_check_points\check_point.ckpt-20
Training // Epoch 1 //  Total r_cost = 5047316.12256  Total s_cost = 327312.91584  Total s_mean = -7.60876  Total s_std = 182.78925  Total a_cost = 0.00000 Training time : 26670 ms negative Sampling time : 12739 ms negative samples : 201770
precision_1	[0.0358101],	||	 precision_5	[0.0285850],	||	 precision_10	[0.0245307],	||	 precision_15	[0.0223589]
recall_1   	[0.0161647],	||	 recall_5   	[0.0653450],	||	 recall_10   	[0.1134043],	||	 recall_15   	[0.1580776]
f_measure_1	[0.0222746],	||	 f_measure_5	[0.0397718],	||	 f_measure_10	[0.0403362],	||	 f_measure_15	[0.0391766]
ndcg_1     	[0.0358101],	||	 ndcg_5     	[0.0538884],	||	 ndcg_10     	[0.0676796],	||	 ndcg_15     	[0.0793029]
Metrics for user type	 Small
precision_1	[0.0424794],	||	 precision_5	[0.0337668],	||	 precision_10	[0.0292805],	||	 precision_15	[0.0265135]
recall_1	[0.0169771],	||	 recall_5	[0.067281

boucle adversarial fin
boucle adversarial debut-- num batch  104
boucle adversarial fin
boucle adversarial debut-- num batch  105
boucle adversarial fin
boucle adversarial debut-- num batch  106
boucle adversarial fin
boucle adversarial debut-- num batch  107
boucle adversarial fin
boucle adversarial debut-- num batch  108
boucle adversarial fin
boucle adversarial debut-- num batch  109
boucle adversarial fin
boucle adversarial debut-- num batch  110
boucle adversarial fin
boucle adversarial debut-- num batch  111
boucle adversarial fin
boucle adversarial debut-- num batch  112
boucle adversarial fin
boucle adversarial debut-- num batch  113
boucle adversarial fin
boucle adversarial debut-- num batch  114
boucle adversarial fin
boucle adversarial debut-- num batch  115
boucle adversarial fin
boucle adversarial debut-- num batch  116
boucle adversarial fin
boucle adversarial debut-- num batch  117
boucle adversarial fin
boucle adversarial debut-- num batch  118
boucle adversarial fin
bo

boucle adversarial fin
boucle adversarial debut-- num batch  10
boucle adversarial fin
boucle adversarial debut-- num batch  11
boucle adversarial fin
boucle adversarial debut-- num batch  12
boucle adversarial fin
boucle adversarial debut-- num batch  13
boucle adversarial fin
boucle adversarial debut-- num batch  14
boucle adversarial fin
boucle adversarial debut-- num batch  15
boucle adversarial fin
boucle adversarial debut-- num batch  16
boucle adversarial fin
boucle adversarial debut-- num batch  17
boucle adversarial fin
boucle adversarial debut-- num batch  18
boucle adversarial fin
boucle adversarial debut-- num batch  19
boucle adversarial fin
boucle adversarial debut-- num batch  20
boucle adversarial fin
boucle adversarial debut-- num batch  21
boucle adversarial fin
boucle adversarial debut-- num batch  22
boucle adversarial fin
boucle adversarial debut-- num batch  23
boucle adversarial fin
boucle adversarial debut-- num batch  24
boucle adversarial fin
boucle adversaria

boucle adversarial fin
boucle adversarial debut-- num batch  138
boucle adversarial fin
boucle adversarial debut-- num batch  139
boucle adversarial fin
boucle adversarial debut-- num batch  140
boucle adversarial fin
boucle adversarial debut-- num batch  141
boucle adversarial fin
boucle adversarial debut-- num batch  142
boucle adversarial fin
boucle adversarial debut-- num batch  143
boucle adversarial fin
boucle adversarial debut-- num batch  144
boucle adversarial fin
boucle adversarial debut-- num batch  145
boucle adversarial fin
boucle adversarial debut-- num batch  146
boucle adversarial fin
boucle adversarial debut-- num batch  147
boucle adversarial fin
boucle adversarial debut-- num batch  148
boucle adversarial fin
boucle adversarial debut-- num batch  149
boucle adversarial fin
boucle adversarial debut-- num batch  150
boucle adversarial fin
boucle adversarial debut-- num batch  151
boucle adversarial fin
boucle adversarial debut-- num batch  152
boucle adversarial fin
bo

boucle adversarial fin
boucle adversarial debut-- num batch  45
boucle adversarial fin
boucle adversarial debut-- num batch  46
boucle adversarial fin
boucle adversarial debut-- num batch  47
boucle adversarial fin
boucle adversarial debut-- num batch  48
boucle adversarial fin
boucle adversarial debut-- num batch  49
boucle adversarial fin
boucle adversarial debut-- num batch  50
boucle adversarial fin
boucle adversarial debut-- num batch  51
boucle adversarial fin
boucle adversarial debut-- num batch  52
boucle adversarial fin
boucle adversarial debut-- num batch  53
boucle adversarial fin
boucle adversarial debut-- num batch  54
boucle adversarial fin
boucle adversarial debut-- num batch  55
boucle adversarial fin
boucle adversarial debut-- num batch  56
boucle adversarial fin
boucle adversarial debut-- num batch  57
boucle adversarial fin
boucle adversarial debut-- num batch  58
boucle adversarial fin
boucle adversarial debut-- num batch  59
boucle adversarial fin
boucle adversaria

...

boucle adversarial fin
boucle adversarial debut-- num batch  24
boucle adversarial fin
boucle adversarial debut-- num batch  25
boucle adversarial fin
boucle adversarial debut-- num batch  26
boucle adversarial fin
boucle adversarial debut-- num batch  27
boucle adversarial fin
boucle adversarial debut-- num batch  28
boucle adversarial fin
boucle adversarial debut-- num batch  29
boucle adversarial fin
boucle adversarial debut-- num batch  30
boucle adversarial fin
boucle adversarial debut-- num batch  31
boucle adversarial fin
boucle adversarial debut-- num batch  32
boucle adversarial fin
boucle adversarial debut-- num batch  33
boucle adversarial fin
boucle adversarial debut-- num batch  34
boucle adversarial fin
boucle adversarial debut-- num batch  35
boucle adversarial fin
boucle adversarial debut-- num batch  36
boucle adversarial fin
boucle adversarial debut-- num batch  37
boucle adversarial fin
boucle adversarial debut-- num batch  38
boucle adversarial fin
boucle adversaria

boucle adversarial fin
boucle adversarial debut-- num batch  152
boucle adversarial fin
boucle adversarial debut-- num batch  153
boucle adversarial fin
boucle adversarial debut-- num batch  154
boucle adversarial fin
boucle adversarial debut-- num batch  155
boucle adversarial fin
boucle adversarial debut-- num batch  156
boucle adversarial fin
boucle adversarial debut-- num batch  157
boucle adversarial fin
boucle adversarial debut-- num batch  158
boucle adversarial fin
boucle adversarial debut-- num batch  159
boucle adversarial fin
boucle adversarial debut-- num batch  160
boucle adversarial fin
boucle adversarial debut-- num batch  161
boucle adversarial fin
boucle adversarial debut-- num batch  162
boucle adversarial fin
boucle adversarial debut-- num batch  163
boucle adversarial fin
boucle adversarial debut-- num batch  164
boucle adversarial fin
boucle adversarial debut-- num batch  165
boucle adversarial fin
boucle adversarial debut-- num batch  166
boucle adversarial fin
bo

boucle adversarial fin
boucle adversarial debut-- num batch  59
boucle adversarial fin
boucle adversarial debut-- num batch  60
boucle adversarial fin
boucle adversarial debut-- num batch  61
boucle adversarial fin
boucle adversarial debut-- num batch  62
boucle adversarial fin
boucle adversarial debut-- num batch  63
boucle adversarial fin
boucle adversarial debut-- num batch  64
boucle adversarial fin
boucle adversarial debut-- num batch  65
boucle adversarial fin
boucle adversarial debut-- num batch  66
boucle adversarial fin
boucle adversarial debut-- num batch  67
boucle adversarial fin
boucle adversarial debut-- num batch  68
boucle adversarial fin
boucle adversarial debut-- num batch  69
boucle adversarial fin
boucle adversarial debut-- num batch  70
boucle adversarial fin
boucle adversarial debut-- num batch  71
boucle adversarial fin
boucle adversarial debut-- num batch  72
boucle adversarial fin
boucle adversarial debut-- num batch  73
boucle adversarial fin
boucle adversaria

boucle adversarial fin
boucle adversarial debut-- num batch  186
boucle adversarial fin
boucle adversarial debut-- num batch  187
boucle adversarial fin
boucle adversarial debut-- num batch  188
boucle adversarial fin
boucle adversarial debut-- num batch  189
boucle adversarial fin
boucle adversarial debut-- num batch  190
boucle adversarial fin
boucle adversarial debut-- num batch  191
boucle adversarial fin
boucle adversarial debut-- num batch  192
boucle adversarial fin
boucle adversarial debut-- num batch  193
boucle adversarial fin
boucle adversarial debut-- num batch  194
boucle adversarial fin
boucle adversarial debut-- num batch  195
boucle adversarial fin
boucle adversarial debut-- num batch  196
boucle adversarial fin
boucle adversarial debut-- num batch  197
boucle adversarial fin
Training // Epoch 8 //  Total r_cost = 93438.87857  Total s_cost = 104.19465  Total s_mean = -0.00119  Total s_std = 197.89880  Total a_cost = 25.62203 Training time : 404964 ms negative Sampling t

boucle adversarial fin
boucle adversarial debut-- num batch  94
boucle adversarial fin
boucle adversarial debut-- num batch  95
boucle adversarial fin
boucle adversarial debut-- num batch  96
boucle adversarial fin
boucle adversarial debut-- num batch  97
boucle adversarial fin
boucle adversarial debut-- num batch  98
boucle adversarial fin
boucle adversarial debut-- num batch  99
boucle adversarial fin
boucle adversarial debut-- num batch  100
boucle adversarial fin
boucle adversarial debut-- num batch  101
boucle adversarial fin
boucle adversarial debut-- num batch  102
boucle adversarial fin
boucle adversarial debut-- num batch  103
boucle adversarial fin
boucle adversarial debut-- num batch  104
boucle adversarial fin
boucle adversarial debut-- num batch  105
boucle adversarial fin
boucle adversarial debut-- num batch  106
boucle adversarial fin
boucle adversarial debut-- num batch  107
boucle adversarial fin
boucle adversarial debut-- num batch  108
boucle adversarial fin
boucle a

In [ ]:
with open('Rec_' + dataname + '_fairAdvBPR.npy', "wb") as f:
    np.save(f, Rec)


In [ ]:
with open('Rec_' + dataname + '_fairAdvBPR.npy', "rb") as f:
    Recom = np.load(f)
Recom.shape

In [ ]:
[precision, recall, f_score, NDCG] = utility.test_model_all(Recom, test_df, train_df)

utility.test_model_per_user_type(Recom, test_df, train_df, user_type_list, key_type)
auc = utility.auc_per_user(Recom, test_df, train_df)
print("AUC global is: ", auc)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from colour import Color

def savepdf_barplot_color_gradient(ymin = 0.5, ymax = 0.7, whis = 5, start_color='pink',end_color='blue',num_color=5, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    fig = plt.figure()
    gs = fig.add_gridspec(1, 2, hspace=0, wspace=0)
    (ax1, ax2) = gs.subplots(sharex='col', sharey='row')
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    ax1.bar(X_axis, axis_y1, color=colors)
    ax1.hlines(y=axis_y1[0], xmin = 0, xmax = len(axis_x)-1, colors='black', linestyles='--', lw=1)
    
    plt.sca(ax1)
    plt.xticks(X_axis, axis_x, rotation =50)
    #plt.xlabel(xlabel)
    #fig.suptitle(title)
    plt.ylabel(ylabel, fontsize=18)
    plt.rcParams.update({'font.size': 13}) 
    plt.grid()
    
    plt.sca(ax2)
    ax2.boxplot(axis_y1, whis = whis)
    ax1.set_ylim(ymin, ymax)
    
    plt.tight_layout()
    plt.savefig(plot_file)

def savepdf_barplot_color_gradient2(start_color='pink',end_color='blue',num_color=20, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    plt.bar(X_axis, axis_y1, color=colors)
    
    
    plt.xticks(X_axis, axis_x, rotation =70)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid()
    
   # plt.tight_layout()
    plt.savefig(plot_file)